---

# SVM Model Pipeline

*Template khusus SVM. Salin file ini ke `src/experiments/svm/` lalu kerjakan algoritma ini saja.*

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Model Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)

---

# Introduction <a name="1"></a>

---

Notebook ini menjadi ruang kerja end-to-end untuk **SVM**. Pilihan preprocessing dan eksperimen tetap independen dari model lain.

Implementasikan SVM dari nol, tentukan pipeline khusus model ini, bandingkan dengan scikit-learn, lalu siapkan refit dan prediksi test.

## Metric

Metrik utama adalah *Macro F1* pada validation dengan kedua kelas diberi bobot sama.

$$\mathrm{MacroF1}=\frac{1}{2}\sum_{c\in\{0,1\}}\mathrm{F1}_c$$

Gunakan `labels=[0, 1]`, `average='macro'`, dan `zero_division=0`.

## Dataset

Input berasal dari `data/train.csv` dan `data/test.csv`. `is_canceled` adalah target, sedangkan `id` dipertahankan untuk submission dan dikeluarkan dari fitur.

## Approach

Split bersama dibuat lebih dahulu. Eksplorasi target-aware dibatasi ke `train_set`; parameter preprocessing dipelajari hanya dari data training. Isi bagian yang bertanda TODO untuk **satu algoritma**, tanpa mewajibkan teknik yang sama untuk model lain.

---

# Initialization <a name="2"></a>

---

Bagian ini menyiapkan library, konfigurasi satu model, file data, dan split bersama.

## Environment Setup

Uncomment installation only when the environment is missing these libraries.

In [ ]:
# %pip install -q numpy pandas scikit-learn

### Insight

Tambahkan dependency khusus algoritma hanya jika memang diperlukan dan catat versinya.

## Import Libraries

Keep the imports needed by this model and its comparator in this cell.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support

## Settings

In [ ]:
class Settings:
    SEED = 42
    MODEL_NAME = 'svm'
    ROOT_DIR = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'data' / 'train.csv').is_file()), Path.cwd())
    DATA_DIR = ROOT_DIR / 'data'
    TRAIN_PATH = DATA_DIR / 'train.csv'
    TEST_PATH = DATA_DIR / 'test.csv'
    OUTPUT_DIR = ROOT_DIR / 'output'
    MODEL_DIR = OUTPUT_DIR / 'final' / 'model' / MODEL_NAME

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
os.environ['PYTHONHASHSEED'] = str(CFG.SEED)

## Load Dataset

In [ ]:
train_df = pd.read_csv(CFG.TRAIN_PATH, dtype={'id': str})
test_df = pd.read_csv(CFG.TEST_PATH, dtype={'id': str})
print(f'Train: {train_df.shape} | Test: {test_df.shape}')

# Create the holdout before any target-aware exploration.
y_all = train_df['is_canceled'].to_numpy()
if not np.isin(y_all, [0, 1]).all():
    raise ValueError('Target is expected to contain only labels 0 and 1.')
rng = np.random.default_rng(CFG.SEED)
valid_idx = np.sort(np.concatenate([
    rng.permutation(np.flatnonzero(y_all == label))[:int(np.ceil(0.2 * np.sum(y_all == label)))]
    for label in (0, 1)
]))
train_idx = np.setdiff1d(np.arange(len(train_df)), valid_idx)
train_set = train_df.iloc[train_idx].copy()
val_set = train_df.iloc[valid_idx].copy()

### Insight

Split dengan seed 42 menggunakan sekitar 20% dari tiap kelas sebagai validation. Jangan ubah indeks ini saat membandingkan konfigurasi.

---

# Data Understanding <a name="3"></a>

---

Gunakan bagian ini untuk memahami data yang tersedia bagi model ini. Pemeriksaan yang memakai target dilakukan hanya pada `train_set`.

## Describe Data

In [ ]:
feature_columns = [column for column in train_df.columns if column not in {'id', 'is_canceled'}]
train_features = train_set[feature_columns]
feature_profile = pd.DataFrame({
    'dtype': train_features.dtypes.astype(str),
    'missing': train_features.isna().sum(),
    'missing_pct': train_features.isna().mean().mul(100).round(2),
    'unique': train_features.nunique(dropna=True),
})
display(feature_profile)

target_counts = train_set['is_canceled'].value_counts().reindex([0, 1], fill_value=0)
display(pd.DataFrame({'count': target_counts, 'proportion': target_counts / len(train_set)}, index=[0, 1]))

### Insight

Ringkasan ini tidak mengubah data. Gunakan hanya untuk merumuskan eksperimen pipeline model ini, bukan untuk melihat label validation.

## Explore Data

In [ ]:
numeric_features = train_features.select_dtypes(include='number').columns
categorical_features = train_features.select_dtypes(exclude='number').columns

display(train_set.groupby('is_canceled')[numeric_features].median().T.rename(columns={0: 'not_canceled', 1: 'canceled'}))
for column in categorical_features:
    category_summary = train_set.assign(category=train_set[column].fillna('__MISSING__')).groupby('category', dropna=False)['is_canceled'].agg(count='size', cancellation_rate='mean')
    display(category_summary.sort_values('count', ascending=False))

### Insight

Target-aware summaries use `train_set` only. Record the hypotheses they suggest and test those choices on the agreed validation split.

## Verify Data Quality

In [ ]:
test_features = test_df.drop(columns='id')
quality_checks = {
    'unique IDs in train': train_df['id'].is_unique,
    'unique IDs in test': test_df['id'].is_unique,
    'train and test feature columns match': set(feature_columns) == set(test_features.columns),
    'binary training target': set(train_set['is_canceled'].unique()).issubset({0, 1}),
}
display(pd.Series(quality_checks, name='passed').to_frame())
if not all(quality_checks.values()):
    raise ValueError('Review data quality checks before continuing.')

### Insight

Perbaiki atau jelaskan temuan pada branch ini; jangan menghapus baris atau memperbaiki nilai tanpa alasan yang dicatat.

---

# Training and Validation Split <a name="4"></a>

---

Split sudah dibuat di Initialization sebelum eksplorasi target-aware agar `train_set` dapat digunakan pada bagian Data Understanding.

In [ ]:
split_class_counts = pd.DataFrame({
    'train': train_set['is_canceled'].value_counts().reindex([0, 1], fill_value=0),
    'validation': val_set['is_canceled'].value_counts().reindex([0, 1], fill_value=0),
})
display(split_class_counts)

cross_split_duplicates = train_set[feature_columns].drop_duplicates().merge(
    val_set[feature_columns].drop_duplicates(), on=feature_columns, how='inner'
).shape[0]
print(f'Identical feature rows across train and validation: {cross_split_duplicates}')

### Insight

Laporkan jumlah contoh per kelas dan duplikasi lintas split. Jangan mengubah split setelah melihat skor validation.

---

# Data Preparation <a name="5"></a>

---

Bagian ini mencatat keputusan preparation untuk algoritma dalam salinan notebook ini. Keputusan tidak diwariskan dari pipeline model lain.

## Select Data

In [ ]:
X_train_raw = train_set[feature_columns].copy()
X_val_raw = val_set[feature_columns].copy()
X_test_raw = test_df[feature_columns].copy()
y_train = train_set['is_canceled'].to_numpy(dtype=int)
y_val = val_set['is_canceled'].to_numpy(dtype=int)

if X_train_raw.columns.tolist() != X_val_raw.columns.tolist() or X_train_raw.columns.tolist() != X_test_raw.columns.tolist():
    raise ValueError('Train, validation, and test feature columns must match in the same order.')
if {'id', 'is_canceled'} & set(feature_columns):
    raise ValueError('ID and target must not be used as model features.')
print(f'Features: {len(feature_columns)} | train: {len(X_train_raw)} | validation: {len(X_val_raw)} | test: {len(X_test_raw)}')

### Insight

`id` stays available separately for submission. Keep feature row order stable throughout preprocessing and prediction.

## Clean Data

### Missing Values

In [ ]:
missing_summary = pd.DataFrame({
    'train': X_train_raw.isna().sum(),
    'validation': X_val_raw.isna().sum(),
    'test': X_test_raw.isna().sum(),
})
display(missing_summary.loc[missing_summary.any(axis=1)].sort_values('train', ascending=False))

### Insight

Pilih strategi missing-value handling untuk model ini di Model Pipeline. Fit nilai imputasi hanya pada bagian training yang sedang digunakan.

### Duplicates

In [ ]:
duplicate_mask = X_train_raw.duplicated(subset=feature_columns, keep=False)
print(f'Training rows with duplicate features: {int(duplicate_mask.sum())}')

### Insight

Duplikasi dilaporkan tetapi tidak dihapus otomatis, terutama bila label pada fitur identik berbeda.

### Outliers

In [ ]:
numeric_columns = X_train_raw.select_dtypes(include='number').columns
quartiles = X_train_raw[numeric_columns].quantile([0.25, 0.75])
iqr = quartiles.loc[0.75] - quartiles.loc[0.25]
lower, upper = quartiles.loc[0.25] - 1.5 * iqr, quartiles.loc[0.75] + 1.5 * iqr
outlier_counts = ((X_train_raw[numeric_columns] < lower) | (X_train_raw[numeric_columns] > upper)).sum()
display(outlier_counts.rename('iqr_outliers').to_frame().sort_values('iqr_outliers', ascending=False))

### Insight

Batas IQR hanya untuk diagnosis. Tentukan penanganannya berdasarkan algoritma dan validasi, bukan dengan clipping otomatis.

### Data Errors

Catat nilai di luar domain, tipe yang tidak sesuai, atau ketidakkonsistenan yang relevan untuk model ini sebelum mengubah data.

## Construct Data

### Attribute Construction

TODO: tuliskan fitur turunan yang akan diuji untuk satu model ini dan alasan pemilihannya.

### Feature Scaling

TODO: putuskan apakah algoritma ini memerlukan scaling dan fitur mana yang diubah.

### Encoding

TODO: pilih representasi categorical features sesuai implementasi model. CART dapat mempertahankan input raw jika model menangani kategori dan null secara native.

### Dimensionality Reduction

TODO: catat apakah seleksi fitur atau reduksi dimensi diperlukan; setiap parameter dipelajari dari training data saja.

## Class Balance

Distribusi kelas tersedia pada split. Jika branch ini memakai class weighting atau resampling, pelajari bobot hanya dari training fold dan lakukan resampling pada training saja.

## Format Data

Tentukan bentuk input yang dibutuhkan satu model ini. Pertahankan jumlah baris serta urutannya ketika membuat matriks model.

---

# Model Pipeline <a name="6"></a>

---

Isi satu pipeline untuk satu algoritma. Semua langkah yang mempelajari state di-*fit* pada training, lalu validation dan test hanya ditransformasi. Untuk CART, `None` dapat dipakai sebagai jalur raw jika implementasinya sendiri mendukung kategori dan null.

In [ ]:
def fit_preprocessor(X_fit):
    """Fit this model's preparation using X_fit only and return the fitted state."""
    raise NotImplementedError('Implement this model branch preparation, or return None for a supported raw-input path.')


def transform_features(preprocessor, X):
    """Transform without fitting on X; preserve the raw frame when state is None."""
    if preprocessor is None:
        return X.copy()
    return preprocessor.transform(X)

preprocessor = fit_preprocessor(X_train_raw)
X_train_model = transform_features(preprocessor, X_train_raw)
X_val_model = transform_features(preprocessor, X_val_raw)

if len(X_train_model) != len(y_train) or len(X_val_model) != len(y_val):
    raise ValueError('Preprocessing must preserve row count and order.')

### Insight

Jangan fit ulang preprocessing pada validation. Bila tuning memakai cross-validation, fit semua preprocessing di dalam setiap fold.

---

# Modeling and Validation <a name="7"></a>

---

Implementasikan dan evaluasi hanya algoritma yang menjadi tujuan salinan notebook ini. Model inti wajib dibuat dari nol.

In [ ]:
svm_configs = {
    'linear': {'kernel': 'linear'},
    'polynomial': {'kernel': 'poly', 'degree': 2},
    'Gaussian': {'kernel': 'rbf'},
}
selected_config = {'kernel': 'linear', 'alpha': 0.01}

def train_model(X, y, config):
    """Train SVM from scratch and return the fitted model."""
    raise NotImplementedError(
        'Implement a binary hinge-loss SVM from scratch for the selected linear, polynomial, or Gaussian kernel.'
    )


def predict_model(model, X):
    """Return one integer label 0 or 1 per input row."""
    raise NotImplementedError('Connect this model branch prediction.')

### Insight

Catat varian model, preprocessing, hyperparameter, seed, dan skor validation yang digunakan untuk memilih konfigurasi.

In [ ]:
model = train_model(X_train_model, y_train, selected_config)
y_val_pred = np.asarray(predict_model(model, X_val_model))
if y_val_pred.ndim != 1 or len(y_val_pred) != len(y_val):
    raise ValueError('Validation prediction must contain one label per validation row.')
if not np.isin(y_val_pred, [0, 1]).all():
    raise ValueError('Predictions must be labels 0 or 1.')
y_val_pred = y_val_pred.astype(int)

validation_macro_f1 = f1_score(y_val, y_val_pred, labels=[0, 1], average='macro', zero_division=0)
validation_confusion = confusion_matrix(y_val, y_val_pred, labels=[0, 1])
precision, recall, class_f1, support = precision_recall_fscore_support(y_val, y_val_pred, labels=[0, 1], zero_division=0)
print(f'Validation Macro F1: {validation_macro_f1:.4f}')
display(pd.DataFrame({'precision': precision, 'recall': recall, 'f1': class_f1, 'support': support}, index=[0, 1]))
display(pd.DataFrame(validation_confusion, index=[0, 1], columns=[0, 1]))

### Insight

Pilih konfigurasi berdasarkan validation yang sama. Jangan gunakan test untuk memilih model atau preprocessing.

---

# Comparison with scikit-learn <a name="8"></a>

---

Gunakan estimator `scikit-learn` sebagai pembanding pada `valid_idx` yang sama. Pembanding dan preprocessing-nya tidak menggantikan implementasi model dari nol.

In [ ]:
def fit_comparison_preprocessor(X_fit):
    """Fit optional comparator preprocessing on training rows only."""
    raise NotImplementedError(
        'Implement comparator preprocessing or return None for raw input.'
    )


def transform_comparison_features(preprocessor, X):
    if preprocessor is None:
        return X.copy()
    return preprocessor.transform(X)


def build_comparison_model(config):
    """Build the scikit-learn SVM comparator."""
    raise NotImplementedError('Return SVC configured with the matching kernel.')

comparison_config = None  # Set the matching comparator configuration.
if comparison_config is None:
    raise NotImplementedError('Set comparison_config for this model.')
comparison_preprocessor = fit_comparison_preprocessor(X_train_raw)
X_train_comparison = transform_comparison_features(
    comparison_preprocessor, X_train_raw
)
X_val_comparison = transform_comparison_features(
    comparison_preprocessor, X_val_raw
)
comparison_model = build_comparison_model(comparison_config)
comparison_model.fit(X_train_comparison, y_train)
y_comparison_pred = np.asarray(comparison_model.predict(X_val_comparison))
if y_comparison_pred.shape != y_val.shape or not np.isin(y_comparison_pred, [0, 1]).all():
    raise ValueError('Comparator must return one binary label per validation row.')
comparison_macro_f1 = f1_score(
    y_val, y_comparison_pred, labels=[0, 1], average='macro', zero_division=0
)
print(f'scikit-learn comparator Macro F1: {comparison_macro_f1:.4f}')

### Insight

Laporkan skor pembanding berdampingan dengan skor model dari nol; keduanya menggunakan baris validation yang sama.

---

# Saving and Loading Models <a name="9"></a>

---

Setelah konfigurasi dipilih, fit ulang preparation dan model pada seluruh `train_df`. Simpan keduanya bersama-sama agar prediksi berikutnya memakai pipeline yang sama.

In [ ]:
X_full_raw = train_df[feature_columns].copy()
y_full = train_df['is_canceled'].to_numpy(dtype=int)
final_preprocessor = fit_preprocessor(X_full_raw)
X_full_model = transform_features(final_preprocessor, X_full_raw)
final_model = train_model(X_full_model, y_full, selected_config)

CFG.MODEL_DIR.mkdir(parents=True, exist_ok=True)
def save_model_bundle(path, preprocessor, model):
    raise NotImplementedError('Save this model and its preprocessing state using the branch-approved format.')

def load_model_bundle(path):
    raise NotImplementedError('Load the saved model and preprocessing state.')

save_model_bundle(CFG.MODEL_DIR, final_preprocessor, final_model)
loaded_preprocessor, loaded_model = load_model_bundle(CFG.MODEL_DIR)

### Insight

Gunakan hanya artifact tepercaya. Uji pemuatan di sesi Python baru sebelum mengandalkan model tersimpan.

---

# Kaggle Submission <a name="10"></a>

---

Prediksi test dibuat dari model dan preprocessing yang baru dimuat, dengan urutan `id` persis seperti `test.csv`.

In [ ]:
X_test_final = transform_features(loaded_preprocessor, X_test_raw)
test_predictions = np.asarray(predict_model(loaded_model, X_test_final))
if test_predictions.ndim != 1 or len(test_predictions) != len(test_df) or not np.isin(test_predictions, [0, 1]).all():
    raise ValueError('Test prediction must contain one label 0 or 1 per test row.')
test_predictions = test_predictions.astype(int)

submission = pd.DataFrame({'id': test_df['id'].to_numpy(), 'is_canceled': test_predictions})
if submission.columns.tolist() != ['id', 'is_canceled'] or not submission['id'].equals(test_df['id'].reset_index(drop=True)):
    raise ValueError('Submission columns, IDs, and row order must match test.csv exactly.')
submission_path = CFG.OUTPUT_DIR / f'submission_{CFG.MODEL_NAME}.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} rows to {submission_path}')

### Insight

Submission berisi hanya `id,is_canceled`, satu prediksi integer untuk setiap baris test.

---

# Error Analysis <a name="11"></a>

---

Analisis kesalahan menggunakan validation yang sudah dievaluasi. Jangan memasukkan label validation kembali ke training model terpilih setelah melihat hasilnya.

In [ ]:
error_analysis = X_val_raw.copy()
error_analysis['actual'] = y_val
error_analysis['predicted'] = y_val_pred
error_analysis['error_type'] = np.select(
    [error_analysis['actual'].eq(0) & error_analysis['predicted'].eq(1),
     error_analysis['actual'].eq(1) & error_analysis['predicted'].eq(0)],
    ['false_positive', 'false_negative'],
    default='correct',
)
display(error_analysis.loc[error_analysis['error_type'] != 'correct'].head(20))
display(error_analysis['error_type'].value_counts())

### Insight

Gunakan pola kesalahan untuk merumuskan eksperimen baru. Jangan memakai hasil test untuk mengubah konfigurasi.